In [1]:
pip install opencv-python facenet-pytorch torch torchvision numpy

  Obtaining dependency information for facenet-pytorch from https://files.pythonhosted.org/packages/ed/2e/2d56386bc2f834cdc683743903852cf1428b4e5ee119f16cf808b589d3cd/facenet_pytorch-2.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/10/58/ed8f7754299f3e91d6414b6dc09f62b3fa7c6e5d63dfe48d69ab81498a37/torchvision-0.26.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/bd/63/05d193dbb4b5eec1eca73822d80da98b511f8328ad4ae3ca4caf0f4db91d/numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata
INFO: pip is looking at multiple versions of facenet-pytorch to determine which version is compatible with other requirements. This could take a while.
  Obtaining dependency information for facenet-pytorch from https://files.pythonhosted.org/packages/40/c1/0a15058f478c8f0f29cacb6e35530b97ead780be99737e62810e71dc477e/facenet_pytorch-2.5.3-py3-none-any.whl.metada

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 2.4.4 which is incompatible.


In [2]:
import cv2
import torch
import torch.nn.functional as F
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

In [3]:
# ---------------------------------------------------------
# Bước 1: Khởi tạo mô hình
# ---------------------------------------------------------
# Sử dụng GPU nếu có, ngược lại dùng CPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [4]:
# Khởi tạo MTCNN để phát hiện và cắt khuôn mặt
# keep_all=False: Chỉ lấy khuôn mặt to/rõ nhất nếu có nhiều mặt
mtcnn = MTCNN(keep_all=False, device=device)

In [5]:
# Khởi tạo FaceNet (InceptionResnetV1) đã được pre-train trên tập vggface2
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

  0%|          | 0.00/107M [00:00<?, ?B/s]

In [ ]:
# ---------------------------------------------------------
# Bước 2: Trích xuất đặc trưng của khuôn mặt mẫu
# ---------------------------------------------------------
reference_image_path = 'my_face.jpg' # Tên file ảnh của bạn
try:
    img_ref = Image.open(reference_image_path)
    # MTCNN cắt khuôn mặt và chuyển thành tensor
    face_ref = mtcnn(img_ref) 
    if face_ref is not None:
        # Thêm chiều batch (1, C, H, W) và đưa vào FaceNet để lấy vector đặc trưng
        emb_ref = resnet(face_ref.unsqueeze(0).to(device)).detach()
        print("Đã tải khuôn mặt mẫu thành công!")
    else:
        print("Không tìm thấy khuôn mặt trong ảnh mẫu.")
        exit()
except Exception as e:
    print(f"Lỗi khi đọc ảnh mẫu: {e}. Vui lòng chuẩn bị file ảnh my_face.jpg")
    exit()

Lỗi khi đọc ảnh mẫu: [Errno 2] No such file or directory: 'my_face.jpg'. Vui lòng chuẩn bị file ảnh my_face.jpg


: 

In [ ]:
# ---------------------------------------------------------
# Bước 3: Thao tác Webcam và Xử lý thời gian thực
# ---------------------------------------------------------
cap = cv2.VideoCapture(0) # Mở webcam mặc định

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Chuyển đổi BGR (OpenCV) sang RGB (MTCNN/PIL yêu cầu)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(frame_rgb)
    
    # Phát hiện khuôn mặt và lấy bounding box (boxes)
    boxes, probs = mtcnn.detect(img_pil)
    
    # Lấy khuôn mặt đã được cắt (tensor) để đưa vào FaceNet
    face_tensor = mtcnn(img_pil)

    if face_tensor is not None and boxes is not None:
        # Trích xuất đặc trưng từ webcam
        emb_webcam = resnet(face_tensor.unsqueeze(0).to(device)).detach()
        
        # Tính độ tương đồng (Cosine Similarity)
        # Kết quả nằm trong khoảng [-1, 1]. Càng gần 1 càng giống nhau.
        similarity = F.cosine_similarity(emb_ref, emb_webcam).item()
        
        # Điều kiện so sánh theo yêu cầu bài tập
        if similarity > 0.7:
            label = f"Matched ({similarity:.2f})"
            color = (0, 255, 0) # Màu xanh lá
        else:
            label = f"Unknown ({similarity:.2f})"
            color = (0, 0, 255) # Màu đỏ
            
        # Vẽ bounding box và hiển thị text lên frame
        for box in boxes:
            x1, y1, x2, y2 = [int(b) for b in box]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # Hiển thị cửa sổ Webcam
    cv2.imshow('Face Recognition Thực Hành', frame)
    
    # Nhấn phím 'q' để thoát
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [ ]:
# Dọn dẹp tài nguyên
cap.release()
cv2.destroyAllWindows()